<a href="https://colab.research.google.com/github/yogevsim-bgu/frank-hertz/blob/main/Frank_Hertz_Exp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Basics for Labs A and B

## Required Installs and Imports

In [ ]:
!pip install -q matplotlib iminuit scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 448.2/448.2 kB 13.3 MB/s eta 0:00:00


In [ ]:
# general python packages for classes, warning ,etc.
from abc import ABC, abstractmethod
from typing import Annotated
import warnings

# numpy for numeric calcualtion
import numpy as np
import seaborn as sns
# matplotlib for plotting
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

# iminuit as the minimizer
from iminuit import Minuit, describe
from iminuit.util import make_func_code, describe

# sklearn for goodness estimation methods
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from scipy.stats.distributions import chi2 #this is for chi2 statistics there is likely a way to calculate (in particular p-value) from numpy or sklearn ...

%matplotlib inline
warnings.simplefilter("ignore")

## Class Definitions

We define here a base regressor class that has the responsibility to provide iminuit with the cost function value. It is also responsible to produce the goodness estimation and to plot the result via the show method.

All changes to the way the plots are presented should be modified here.

In [ ]:
class Regressor(ABC):
    """
    Abstract base class for regressors compatible with Iminuit.
    To edit the plotting of the graph, edit the show method.
    """

    @abstractmethod
    def __init__(self, model : callable, x : list, y: list, dx : list, dy : list)->None:
        pass

    #this part defines the calculations when the function is called
    @abstractmethod
    def __call__(self, *par)->float:
        pass

    def goodness_estimator(self,optimizer : callable,method : str):
        """
        Performs goodness estimation given some metric (method).
        outputs 2 value of the goodness and the output string to be displayed.
        """
        self.goodness_methods = ["chi2","mse","mae","r2"]
        y_pred = self.model(self.x, *self.par_values)
        if method not in self.goodness_methods:
            raise ValueError(f"No method of type {method} are available.\n Choose one from {[item for item in self.goodness_methods]}")
        if method == "mse":
            return mean_squared_error(y_pred=y_pred,y_true=self.y) , f"MSE = {mean_squared_error(y_pred=y_pred,y_true=self.y):.4f}"
        if method == "mae":
            return mean_absolute_error(y_pred=y_pred,y_true=self.y) , f"MAE = {mean_absolute_error(y_pred=y_pred,y_true=self.y):.4f}"
        if method == "r2":
            return r2_score(y_pred=y_pred,y_true=self.y) , f"$R^2$ = {r2_score(y_pred=y_pred,y_true=self.y):.4f}"
        if method == "chi2":
            self.par = optimizer.parameters
            self.chi2 = optimizer.fval
            self.ndof = len(self.x) - len(self.par)
            self.chi_ndof = self.chi2 / self.ndof
            return self.chi_ndof , f"$\chi^2$ / ndof = {self.chi_ndof:.4f}({self.chi2:.4f}/{self.ndof})"


    def show(self, optimizer : callable , x_title="X", y_title="Y",plot_title=None, goodness_loc=2,legend_loc='best',show_grid=False,sig_digits=3,metric="chi2"):
        """
        Plots and displays the graph.
        """
        self.par        = optimizer.parameters
        self.par_values = optimizer.values
        self.par_errors = optimizer.errors
        text=""
        for p in (self.par):
            text += rf"{p} = {self.par_values[p]:.{sig_digits}f} $\pm$ {self.par_errors[p]:.{sig_digits}f}" + "\n"
        self.goodness_metric, metric_text = self.goodness_estimator(optimizer,method=metric)
        text = text + metric_text

        self.func_x = np.linspace(self.x[0],self.x[-1] ,10000) # 10000 linearly spaced numbers
        self.y_fit = self.model(self.func_x, *self.par_values)

        plt.rc("font", size=16)
        fig = plt.figure(figsize=(8,6))
        ax = fig.add_axes([0,0,1,1])
        if plot_title is not None:
            ax.set_title(plot_title)
        ax.plot(self.func_x,self.y_fit,label="Fit") #plot the function over 10k points covering the x axis
        ax.scatter(self.x,self.y, c="red")
        ax.errorbar(self.x, self.y, self.dy, self.dx,fmt='none',ecolor='red', capsize=3,label="Data")
        ax.set_xlabel(x_title, fontdict={"size":21})
        ax.set_ylabel(y_title,fontdict={"size":21})
        ax.tick_params(axis="both",labelsize=18)
        anchored_text = AnchoredText(text, loc=goodness_loc)
        ax.add_artist(anchored_text)
        ax.legend()
        plt.grid(show_grid)

    def get_mean_uncertainty(self, optimizer : callable)->float:
        self.par        = optimizer.parameters
        self.par_values = optimizer.values
        self.par_errors = optimizer.errors
        self.y_fit = self.model(self.x, *self.par_values)
        return np.sqrt( ((self.y-self.y_fit)**2).sum()/(self.ndof) )


    def plot_residuals(self, optimizer : callable,x_title="X", y_title="Residuals", plot_lines=False, plot_title=None,):
        self.par        = optimizer.parameters
        self.par_values = optimizer.values
        self.par_errors = optimizer.errors
        self.y_fit = self.model(self.x, *self.par_values)
        self.residuals = self.y - self.y_fit
        residual_mean = np.mean(self.residuals)
        residual_std  = np.std(self.residuals,ddof=len(self.par)) #ddof because that's how much we are taking out with the fit &matches get_mean_uncertainty see https://numpy.org/doc/2.1/reference/generated/numpy.std.html
        plt.rc("font", size=16)
        fig = plt.figure(figsize=(8,6))
        ax = fig.add_axes([0,0,1,1])
        if plot_title is not None:
            ax.set_title(plot_title)
        ax.scatter(self.x,self.residuals, c="black")
        ax.errorbar(self.x, self.residuals, self.dy,fmt='none',ecolor='black')
        ax.set_xlabel(x_title, fontdict={"size":21})
        ax.set_ylabel(y_title,fontdict={"size":21})
        ax.tick_params(axis="both",labelsize=18)
        plt.axhline(y=0, color='black', linestyle='-')
        if (plot_lines == True):
          plt.axhline(y=residual_mean, color='blue', linestyle='--', label=f"Mean: {residual_mean:.3f}")
          plt.axhline(y=residual_mean + residual_std, color='red', linestyle='--', label=f"Mean ± (STD = {residual_std:.3f})")
          plt.axhline(y=residual_mean - residual_std, color='red', linestyle='--')
        ax.legend(loc="upper right", fontsize=14)




The following 2 classes are specific implementations of the base regressor class. The first is the Least Squares Regressor following the least squares algorithm. Notice that this algorithm totally ignore uncertainties!

The 2nd regressor is the Chi2 regressor. It can handle uncertainties in both x and y. If there are no dx, set them all to 0 or input only dy.

In [ ]:
class LeastSquaresReg(Regressor):
    errordef = Minuit.LEAST_SQUARES  # for Minuit to compute errors correctly

    def __init__(self, model, x, y, dx=None, dy=None)->None:
        self.model = model  # model predicts y for given x
        self.x = np.asarray(x)
        self.y = np.asarray(y)
        if dx is not None:
          print("This is least squares regressor. dx will be ignored!")
        if dy is not None:
          print("This is least squares regressor. dy will be ignored!")
        self.dx = np.zeros(x.shape)
        self.dy = np.zeros(y.shape)
        super().__init__(model, x, y, dx, dy)
        pars = describe(model, annotations=True)
        model_args = iter(pars)
        next(model_args)
        self._parameters = {k: pars[k] for k in model_args}


    @property
    def ndata(self):
        return len(self.x)

    def __call__(self, *par)->float:  # we must accept a variable number of model parameters
        ym = self.model(self.x, *par)
        return np.sum((self.y - ym) ** 2)





In [ ]:
class Chi2Reg(Regressor):  #This class is for Chi2 Regression and if dx is provided uses the effective variance method.
    #this part defines the variables the class will use
    def __init__(self, model, x, y, dx=None, dy=None):
        self.model = model  # model predicts y value for given x value
        self.x  = np.asarray(x) #the x values
        self.y  = np.asarray(y) #the y values
        if dx is None:
          self.dx = np.zeros(x.shape)
        else:
          self.dx = np.asarray(dx)
        if dy is None:
          raise ValueError("Uncertainties on y were not provided!")
        self.dy = np.asarray(dy) #the y-axis uncertainties
        self.func_code = make_func_code(describe(self.model)[1:])
        self.h = (x[-1]-x[0])/10000  #this is the step size for the numerical calculation of the df/dx = last value in x (x[-1]) - first value in x (x[0])/10000

    @property
    def ndata(self):
        return len(self.x)


    def __call__(self, *par):  # par are a variable number of model parameters
        self.ym = self.model(self.x, *par)
        df = (self.model(self.x + self.h, *par)-self.ym)/self.h #the derivative df/dx at point x is taken as [f(x+h)-f(x)]/h
        chi2 = sum(((self.y - self.ym)**2)/(self.dy**2+(df * self.dx)**2))#chi2 is now Sum of: (f(x)-y)^2/(uncert_y^2+(df/dx*uncert_x)^2)
        return chi2



## Examples - Using the classes to fit

We start by defining the measurements and uncertainties

Then we define the function to fit

In [ ]:
linear_fun = lambda x,a,b: a + (x * b)
exp_fun    = lambda x,a,b: a*np.exp(x*b)
power_fun  = lambda x,a,b,c: a*x**b + c
parab_fun  = lambda x,a,b,c: a*x**2 + b*x + c


And if we want to see the covariance matrix we can

In [ ]:
import pandas as pd

# Load experiment measurement files explicitly as CSV
exp2m1_df = pd.read_csv('exp2m1.csv')
exp2m2_df = pd.read_csv('exp2m2.csv')


In [ ]:
# Create a list of all experiment dataframes
experiment_dfs = [
    exp2m1_df, exp2m2_df
]
# Rename columns and plot for all dataframes in the list
for i, df in enumerate(experiment_dfs):
    df_name = f'exp1m{i+1}_df'
    # Rename columns
    if 'Voltage (V)' in df.columns and 'Current (a.u.)' in df.columns:
        df.rename(columns={'Voltage (V)': 'voltage', 'Current (a.u.)': 'current'}, inplace=True)
        print(f"Renamed columns in {df_name}.")

    # Plot using the new column names
    print(f"Plotting {df_name}:")
    df.plot(x='voltage', y='current', kind='scatter', title=f'Experiment {i+1}', grid=True)

In [ ]:
parab_min_representation = lambda v, v0, c:  ((v- v0)**2)+c

In [ ]:
def get_points_between_voltages(vi_df,starting_voltage, ending_voltage):
  return vi_df[(vi_df['voltage'] >= starting_voltage) & (vi_df['voltage'] <= ending_voltage)]

In [ ]:
def get_chi2_for_voltage_range(vi_df,starting_voltage, ending_voltage,dI,multiplier, dv=None):
  truncated_data = get_points_between_voltages(vi_df,starting_voltage, ending_voltage)
  if truncated_data.empty:
    #print(f"Skipping start_point={start_point} because truncated_data is empty (range from {start_point} to {best_end_point} is invalid or empty).")
    return None
  v = truncated_data['voltage'].to_numpy() # Convert Series to numpy array
  if len(v) <= 2: # less data than parameters
    return 0
  I = truncated_data['current'].to_numpy()*multiplier # Convert Series to numpy array
  if dv is None:
    dv = [0.01*10**-3] * len(I)



  chi2 = Chi2Reg(parab_min_representation, v, I, dy=[dI]*len(I), dx=dv)
  m = Minuit(chi2, v0=(starting_voltage + ending_voltage)/2, c=0)
  m.migrad()
  current_chi2_ndof = m.fval/(len(v)-m.nfit)
  return current_chi2_ndof, m

In [ ]:
def get_all_posible_ranges_chi2(vi_df,starting_voltage, ending_voltage,dI,multiplier, range=5,):
  results_df = pd.DataFrame(columns=['starting_voltage', 'ending_voltage', 'chi2_ndof'])
  all_voltage_points_in_range = vi_df[(vi_df['voltage'] >= starting_voltage) & (vi_df['voltage'] <= ending_voltage)]['voltage'].unique()
  possible_starts = all_voltage_points_in_range[all_voltage_points_in_range <= starting_voltage+range]

  possible_ends = all_voltage_points_in_range[all_voltage_points_in_range >= ending_voltage-range]

  for starting_point in possible_starts:
    for end_point in possible_ends:
      results_df.loc[len(results_df)] = [starting_point, end_point, get_chi2_for_voltage_range(vi_df,starting_point, end_point,dI, multiplier)]
  return results_df

In [ ]:
def get_best_range(vi_df,starting_voltage, ending_voltage,dI, multiplier,range=5,heatmap=False):
  results_df = get_all_posible_ranges_chi2(vi_df,starting_voltage, ending_voltage,dI,multiplier, range=range)
  if heatmap:
    plt.figure(figsize=(10, 6))
    sns.heatmap(results_df.pivot(index='starting_voltage', columns='ending_voltage', values='chi2_ndof'), annot=False, fmt=".2f", cmap="PuRd_r", vmin=0, vmax=45)
  best_row = results_df.loc[(results_df['chi2_ndof'] -1).abs().idxmin()]
  return best_row

In [ ]:
def calculate_dI(vi_df, multiplier,dead_zone_start=2, dead_zone_end=8):
  dead_data = vi_df[(vi_df['voltage'] >= dead_zone_start) & (vi_df['voltage'] <= dead_zone_end)]
  return dead_data['current'].std() * multiplier

In [ ]:
def get_best_fit(vi_df,starting_voltage, ending_voltage,dI,multiplier, range=5):
  results_df = get_all_posible_ranges_chi2(vi_df,starting_voltage, ending_voltage,dI,multiplier, range=range)
  best_row = results_df.loc[(results_df['chi2_ndof'] -1).abs().idxmin()]
  best_start = best_row['starting_voltage']
  best_end = best_row['ending_voltage']
  m = best_row['m']
  return m, best_start, best_end

In [ ]:
dI1 = calculate_dI(exp2m1_df,50)
dI2 = calculate_dI(exp2m2_df,1)


In [ ]:
m, _ ,_ = get_best_fit(exp2m1_df,11,15,dI1,50, range=1)